In [17]:
from langchain_openai import ChatOpenAI
from langchain.agents import create_agent, AgentState
from langchain.agents.middleware import before_model,after_model
from rich import print as rprint
from langgraph.runtime import Runtime
from langchain.messages import  RemoveMessage
from langgraph.graph.message import  REMOVE_ALL_MESSAGES
from langgraph.checkpoint.memory import InMemorySaver
from langchain_core.messages import SystemMessage, HumanMessage, AIMessage
from langgraph.store.memory import InMemoryStore


model = ChatOpenAI(
    model="MiniMax-M3",
    api_key='sk-cp-GsTRU_GlFRX_WPY8cf8IqBTDtrVJP4zl4EWLZ8D7627lrDtV-MbWYnWWK7rmvT7ZRluuMtUL7VP2sCD2uq6SzkpIlY0EAJtCIHW2uSGAT4QKYxX9mJQmK8I',
    base_url="https://api.minimaxi.com/v1",
)

agent = create_agent(
    model=model,
)

store = InMemoryStore()

namespace = ("users",)
user_id = "user_1"
user_name = "小明"

store.put(namespace, key = user_id, value = {"name": user_name})

item = store.get(namespace, user_id)
rprint(item)

print("=== 记忆治理策略：消息裁剪 ===")
store.put(namespace, key="user_2", value={"name": "小ming"})
item = store.get(namespace, "user_2")
rprint(item)


Item(namespace=['users'], key='user_1', value={'name': '小明'}, created_at='2026-10-13T07:42:27.955198+00:00', 
updated_at='2026-10-13T07:42:27.955202+00:00')

=== 记忆治理策略：消息裁剪 ===


Item(namespace=['users'], key='user_2', value={'name': '小ming'}, created_at='2026-10-13T07:42:27.958031+00:00', 
updated_at='2026-10-13T07:42:27.958033+00:00')

In [38]:
# search的使用
namespace1 = ("users","evan")
namespace2 = ("users","tino")
namespace3 = ("users","uu")

key1 = "user_1"
key2 = "user_2"
key3 = "user_3"

value1 = {"course": "数学一", "score": 90}
value2 = {"course": "数学二", "score": 85}
value3 = {"course": "数学三", "score": 95}

store.put(namespace1, key=key1, value=value1)
store.put(namespace2, key=key2, value=value2)
store.put(namespace3, key=key3, value=value3)

res = store.search(("users",),filter={"score": 85})
rprint(res)

[
    Item(namespace=['users', 'tino'], key='user_2', value={'course': '数学二', 'score': 85}, 
created_at='2026-10-13T08:05:03.815440+00:00', updated_at='2026-10-13T08:05:03.815440+00:00', score=None)
]

In [54]:
def embed(text):
    return [[1.0]*6 for _ in range(len(text))]

index_config={
    "embed":embed,
    "dims":6,
    "fields":["$","course"]
}
store_index = InMemoryStore(index=index_config)

namespace4 = ("users","yuki")
key4 = "user_4"
value4 = {"course": "英语一", "score": 88}

namespace5 = ("users","gina")
key5 = "user_5"
value5 = {"course": "英语二", "score": 92}

namespace6 = ("users","lily")
key6 = "user_6"
value6 = {"course": "英语三", "score": 85}

store_index.put(namespace4, key=key4, value=value4)
store_index.put(namespace5, key=key5, value=value5)
store_index.put(namespace6, key=key6, value=value6)
rprint(store_index._vectors[("users","yuki")])

defaultdict(<class 'dict'>, {
    'user_4': {'$': [1.0, 1.0, 1.0, 1.0, 1.0, 1.0], 'course': [1.0, 1.0, 1.0, 1.0, 1.0, 1.0]}
})